In [1]:
import requests, lxml
from bs4 import BeautifulSoup

In [2]:
r = requests.get('https://grownyc.org/locations/?order=title-asc&location-type=greenmarket%2cfarmstand&amenity=textiles-recycling&paging=1')

In [3]:
r

<Response [200]>

In [4]:
source = r.content

In [5]:
soup = BeautifulSoup(source, 'html.parser')

In [6]:
locationsTemp=soup.find_all('h3', class_='regular-text')

In [7]:
locations=[]
for i in locationsTemp:
    locations.append(i.text)

In [8]:
locations

['79th Street Greenmarket\n\n',
 'Carroll Gardens Greenmarket\n\n',
 'Cortelyou Greenmarket\n\n',
 'Fort Greene Greenmarket\n\n',
 'Grand Army Plaza Greenmarket\n\n',
 'McCarren Park Greenmarket\n\n',
 'Sunnyside Greenmarket\n\n',
 'Tompkins Square Greenmarket\n\n',
 'Tribeca Greenmarket\n\n']

In [9]:
newLocations=[]

for i in locations:
    new = i.strip()
    newLocations.append(new)

In [10]:
newLocations

['79th Street Greenmarket',
 'Carroll Gardens Greenmarket',
 'Cortelyou Greenmarket',
 'Fort Greene Greenmarket',
 'Grand Army Plaza Greenmarket',
 'McCarren Park Greenmarket',
 'Sunnyside Greenmarket',
 'Tompkins Square Greenmarket',
 'Tribeca Greenmarket']

In [11]:
from geopy.geocoders import Nominatim

In [21]:
geolocator = Nominatim(user_agent='nyc_greenmarket_locator')
textiles = []

for i in newLocations:
    location1 = geolocator.geocode(i)
    if location1:
        textiles.append({
            'Sitename': i,
            'Latitude': location1.latitude,
            'Longitude': location1.longitude,
            'Full Address': location1.address
        })
    else:
        textiles.append({
            'Sitename': i,
            'Latitude': None,
            'Longitude': None,
            'Full Address': f'No address found for {i}'
        })

textiles

[{'Sitename': '79th Street Greenmarket',
  'Latitude': 40.7543845,
  'Longitude': -73.8879886,
  'Full Address': 'Jackson Heights Greenmarket, 79th Street, Jackson Heights, Queens, Queens County, New York, 11373, United States'},
 {'Sitename': 'Carroll Gardens Greenmarket',
  'Latitude': 40.680765,
  'Longitude': -73.9955654,
  'Full Address': 'Carroll Gardens Greenmarket, Carroll Street, Carroll Gardens, Brooklyn, Kings County, New York, 11231, United States'},
 {'Sitename': 'Cortelyou Greenmarket',
  'Latitude': None,
  'Longitude': None,
  'Full Address': 'No address found for Cortelyou Greenmarket'},
 {'Sitename': 'Fort Greene Greenmarket',
  'Latitude': 40.6898009,
  'Longitude': -73.9732851,
  'Full Address': 'Fort Greene Park Greenmarket, Washington Park, Fort Greene, Brooklyn, Kings County, New York, 11205, United States'},
 {'Sitename': 'Grand Army Plaza Greenmarket',
  'Latitude': 40.6724077,
  'Longitude': -73.9699763,
  'Full Address': 'Grand Army Plaza Greenmarket, West Dr

In [22]:
for i in textiles:
    i['Category']='Textiles'

In [27]:
import pandas as pd
data = textiles

df = pd.DataFrame(data)
df.index.name = 'Index'
df.to_csv('textiles.csv', index=False)

df

,Sitename,Latitude,Longitude,Full Address,Category
Index,,,,,
0,79th Street Greenmarket,40.754385,-73.887989,"Jackson Heights Greenmarket, 79th Street, Jack...",Textiles
1,Carroll Gardens Greenmarket,40.680765,-73.995565,"Carroll Gardens Greenmarket, Carroll Street, C...",Textiles
2,Cortelyou Greenmarket,NaN,NaN,No address found for Cortelyou Greenmarket,Textiles
3,Fort Greene Greenmarket,40.689801,-73.973285,"Fort Greene Park Greenmarket, Washington Park,...",Textiles
4,Grand Army Plaza Greenmarket,40.672408,-73.969976,"Grand Army Plaza Greenmarket, West Drive, Broo...",Textiles
5,McCarren Park Greenmarket,40.719717,-73.952612,"Greenpoint / McCarren Park Greenmarket, Union ...",Textiles
6,Sunnyside Greenmarket,NaN,NaN,No address found for Sunnyside Greenmarket,Textiles
7,Tompkins Square Greenmarket,NaN,NaN,No address found for Tompkins Square Greenmarket,Textiles
8,Tribeca Greenmarket,NaN,NaN,No address found for Tribeca Greenmarket,Textiles


In [33]:
import csv

dfT = pd.read_csv('textiles_updated.csv')

dfT

,Sitename,Latitude,Longitude,Address,Borough,Zip,Category
0,79th Street Greenmarket,40.782150,-73.975970,79th Street & Columbus Ave,Manhattan,10024,Textiles
1,Carroll Gardens Greenmarket,40.680765,-73.995565,Carroll St,Brooklyn,11231,Textiles
2,Cortelyou Greenmarket,40.641510,-73.965670,Cortelyou Road,Brooklyn,11226,Textiles
3,Fort Greene Greenmarket,40.689801,-73.973285,Washington Park & Dekalb Ave,Brooklyn,11205,Textiles
4,Grand Army Plaza Greenmarket,40.672408,-73.969976,Prospect Park West & Flatbush Ave,Brooklyn,11238,Textiles
5,McCarren Park Greenmarket,40.719717,-73.952612,Union Ave & N 12th St,Brooklyn,11211,Textiles
6,Sunnyside Greenmarket,40.747120,-73.920570,Skillman Ave,Queens,11104,Textiles
7,Tompkins Square Greenmarket,40.726670,-73.981680,Avenue A & E 7th St,Manhattan,10009,Textiles
8,Tribeca Greenmarket,40.717020,-74.010430,Greenwich St & Reade St,Manhattan,10013,Textiles


In [36]:
import folium

textile_map = folium.Map(location=[dfT['Latitude'].mean(), dfT['Longitude'].mean()], zoom_start=10)
for Sitename, row in dfT.iterrows():
    folium.Marker(
        location=[row['Latitude'], row['Longitude']],
        popup=row['Address'],
        tooltip=row['Sitename']
    ).add_to(textile_map)

textile_map